In [1]:
import matplotlib.pyplot as plt
import pickle

from pmbrl.model2 import Model
from pmbrl.data import Experiment_Data, get_data_expanded

In [2]:
nome_do_arquivo = 'regular_normilized.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    model = exp['model']

del exp
del arquivo

In [3]:
data.evaluate_model(model, path='../testing_data.csv')
results = data.get_evaluation_metrics()
results.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,rse_s1,rse_s2,rse_s3,rse_r,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized,rse,rse_normalized
0,0,0,"(0.5280280669895349, 0.1691488904272192)","(-0.018, -0.046, 0.013, -0.019)",0,1.0,"(-0.019, -0.285, 0.013, 1.258)",0.0,1.0,"(-0.025, -0.525, 0.038, 2.534)",...,0.041,0.003,0.901,0.0,0.546990,0.536136,0.508393,0.511765,0.946,0.525821
1,1,0,"(0.5280280669895349, 0.1691488904272192)","(-0.019, -0.285, 0.013, 1.258)",0,1.0,"(-0.025, -0.525, 0.038, 2.534)",1.0,1.0,"(-0.035, -0.288, 0.088, 1.326)",...,0.067,0.007,1.008,0.0,0.548046,0.542527,0.517986,0.520921,1.084,0.532370
2,2,0,"(0.5280280669895349, 0.1691488904272192)","(-0.025, -0.525, 0.038, 2.534)",1,1.0,"(-0.035, -0.288, 0.088, 1.326)",0.0,1.0,"(-0.041, -0.532, 0.115, 2.694)",...,0.041,0.003,0.968,0.0,0.550158,0.536136,0.508393,0.517498,1.016,0.528046
3,3,0,"(0.5280280669895349, 0.1691488904272192)","(-0.035, -0.288, 0.088, 1.326)",0,1.0,"(-0.041, -0.532, 0.115, 2.694)",1.0,1.0,"(-0.052, -0.3, 0.169, 1.601)",...,0.052,0.001,0.917,0.0,0.552270,0.538840,0.503597,0.513134,0.976,0.526960
4,4,0,"(0.5280280669895349, 0.1691488904272192)","(-0.041, -0.532, 0.115, 2.694)",1,1.0,"(-0.052, -0.3, 0.169, 1.601)",0.0,1.0,"(-0.058, -0.546, 0.201, 3.045)",...,0.042,0.002,1.012,0.0,0.555438,0.536382,0.505995,0.521263,1.065,0.529769


In [4]:
expansions = {
    'estimated_p': ['p0', 'p1'],
    's': ['s0', 's1', 's2', 's3'],
    's_': ['s_0', 's_1', 's_2', 's_3'],
    's__': ['s__0', 's__1', 's__2', 's__3'],
}


# df = data.evaluation_data[data.evaluation_data['episode'] == 98].copy().reset_index(drop=True)
df = data.evaluation_data.copy()
df = get_data_expanded(df, expansions)
df.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,s2,s3,s_0,s_1,s_2,s_3,s__0,s__1,s__2,s__3
0,0,0,"(0.5280280669895349, 0.1691488904272192)","(-0.018, -0.046, 0.013, -0.019)",0,1.0,"(-0.019, -0.285, 0.013, 1.258)",0.0,1.0,"(-0.025, -0.525, 0.038, 2.534)",...,0.013,-0.019,-0.019,-0.285,0.013,1.258,-0.025,-0.525,0.038,2.534
1,1,0,"(0.5280280669895349, 0.1691488904272192)","(-0.019, -0.285, 0.013, 1.258)",0,1.0,"(-0.025, -0.525, 0.038, 2.534)",1.0,1.0,"(-0.035, -0.288, 0.088, 1.326)",...,0.013,1.258,-0.025,-0.525,0.038,2.534,-0.035,-0.288,0.088,1.326
2,2,0,"(0.5280280669895349, 0.1691488904272192)","(-0.025, -0.525, 0.038, 2.534)",1,1.0,"(-0.035, -0.288, 0.088, 1.326)",0.0,1.0,"(-0.041, -0.532, 0.115, 2.694)",...,0.038,2.534,-0.035,-0.288,0.088,1.326,-0.041,-0.532,0.115,2.694
3,3,0,"(0.5280280669895349, 0.1691488904272192)","(-0.035, -0.288, 0.088, 1.326)",0,1.0,"(-0.041, -0.532, 0.115, 2.694)",1.0,1.0,"(-0.052, -0.3, 0.169, 1.601)",...,0.088,1.326,-0.041,-0.532,0.115,2.694,-0.052,-0.300,0.169,1.601
4,4,0,"(0.5280280669895349, 0.1691488904272192)","(-0.041, -0.532, 0.115, 2.694)",1,1.0,"(-0.052, -0.3, 0.169, 1.601)",0.0,1.0,"(-0.058, -0.546, 0.201, 3.045)",...,0.115,2.694,-0.052,-0.300,0.169,1.601,-0.058,-0.546,0.201,3.045


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

def find_params(m, inputs, target, initial_params, num_epochs=500, learning_rate=.1):
    input_values = torch.tensor(inputs).reshape(1, len(inputs))
    param = torch.tensor(initial_params, requires_grad=True)
    target_value = torch.tensor(target).reshape(1, len(target))

    optimizer = optim.Adam([param], lr=learning_rate)
    criterion = nn.MSELoss(reduction='none')

    history = []

    for p in m.parameters():
        p.requires_grad = False

    for epoch in range(num_epochs):
        # Forward pass
        state_inputs = torch.concat([input_values, param.reshape(1, len(initial_params))], dim=1)
        output = m(state_inputs.float())  # Add batch dimension

        # Calculate the loss
        open_loss = criterion(output.float(), target_value.float())
        loss = torch.sqrt(open_loss.sum(axis=1).mean())
        

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        history.append((param.tolist(), output.tolist()[0], loss.item()))

        if (epoch + 1) % 100 == 0:
            # print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {param.item():.4f}, Output: {output.item():.4f}')
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {[round(p,4) for p in param.tolist()]}')
        return history


In [6]:
def predict(row):
    m = model.transition_estimator.state_layer
    inputs = [row.s0, row.s1, row.s2, row.s3, row.a]
    targets = [row.s_0, row.s_1, row.s_2, row.s_3]
    init_param = [row.p0, row.p1]

    hist = find_params(m, inputs, targets, init_param)
    return hist[-1]


In [7]:
df[['new_estimated_params', 'new_estimated_s', 'param_rse']] = df.apply(lambda row: predict(row), axis=1, result_type='expand')

In [9]:
expansions = {
    'new_estimated_params': ['p0', 'p1'],
    's': ['s_0', 's_1', 's_2', 's_3'],
    's_': ['s__0', 's__1', 's__2', 's__3'],
}

final_df = data.evaluation_data.copy()
final_df = get_data_expanded(df, expansions)
final_df.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,s_1,s_2,s_3,s__0,s__1,s__2,s__3,new_estimated_params,new_estimated_s,param_rse
0,0,0,"(0.5280280669895349, 0.1691488904272192)","(-0.018, -0.046, 0.013, -0.019)",0,1.0,"(-0.019, -0.285, 0.013, 1.258)",0.0,1.0,"(-0.025, -0.525, 0.038, 2.534)",...,-0.046,0.013,-0.019,-0.019,-0.285,0.013,1.258,"[-0.04600013047456741, 0.046000003814697266]","[-0.010936334729194641, -0.25009414553642273, ...",0.952140
1,1,0,"(0.5280280669895349, 0.1691488904272192)","(-0.019, -0.285, 0.013, 1.258)",0,1.0,"(-0.025, -0.525, 0.038, 2.534)",1.0,1.0,"(-0.035, -0.288, 0.088, 1.326)",...,-0.285,0.013,1.258,-0.025,-0.525,0.038,2.534,"[-0.016000032424926758, 0.04200001060962677]","[-0.01429833471775055, -0.48695483803749084, 0...",0.899598
2,2,0,"(0.5280280669895349, 0.1691488904272192)","(-0.025, -0.525, 0.038, 2.534)",1,1.0,"(-0.035, -0.288, 0.088, 1.326)",0.0,1.0,"(-0.041, -0.532, 0.115, 2.694)",...,-0.525,0.038,2.534,-0.035,-0.288,0.088,1.326,"[-0.12999998033046722, 0.3409999907016754]","[-0.04941017925739288, -0.3443355858325958, 0....",0.987243
3,3,0,"(0.5280280669895349, 0.1691488904272192)","(-0.035, -0.288, 0.088, 1.326)",0,1.0,"(-0.041, -0.532, 0.115, 2.694)",1.0,1.0,"(-0.052, -0.3, 0.169, 1.601)",...,-0.288,0.088,1.326,-0.041,-0.532,0.115,2.694,"[0.08799983561038971, 0.13500002026557922]","[-0.02843053638935089, -0.49332013726234436, 0...",0.966699
4,4,0,"(0.5280280669895349, 0.1691488904272192)","(-0.041, -0.532, 0.115, 2.694)",1,1.0,"(-0.052, -0.3, 0.169, 1.601)",0.0,1.0,"(-0.058, -0.546, 0.201, 3.045)",...,-0.532,0.115,2.694,-0.052,-0.300,0.169,1.601,"[-0.024999968707561493, 0.4309999942779541]","[-0.057288870215415955, -0.34078308939933777, ...",0.886807


In [ ]:
# FAZER INFERENCIA DE S__ USANDO OS PARAMETROS OPTIMIZADOS E CALCULAR OS RMSE's

In [10]:
final_df[['rse', 'rse_normalized', 
       'rse_s0', 'rse_s1', 'rse_s2', 'rse_s3', 'rse_r', 'rse_s0_normalized',
       'rse_s1_normalized', 'rse_s2_normalized', 'rse_s3_normalized']].describe()

KeyError: "None of [Index(['rse', 'rse_normalized', 'rse_s0', 'rse_s1', 'rse_s2', 'rse_s3',\n       'rse_r', 'rse_s0_normalized', 'rse_s1_normalized', 'rse_s2_normalized',\n       'rse_s3_normalized'],\n      dtype='object')] are in the [columns]"

In [ ]:
del model
del data